# Word Embeddings

## Learning Objectives
1. Build skip-gram embeddings from scratch with numpy SGD.
2. Implement Word2Vec negative sampling in PyTorch.
3. Evaluate embeddings via analogy tasks and nearest-neighbour retrieval.
4. Visualize embedding geometry with t-SNE and compare random vs trained embeddings.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Level 1: Skip-Gram from Scratch (numpy)

Skip-gram predicts surrounding context words given a center word.
For each (center, context) pair we update two embedding matrices W_in and W_out.
Loss = -log P(context | center) = -log sigmoid(W_out[context] · W_in[center]).

In [ ]:
# ── Synthetic text corpus (20 sentences) ────────────────────────────
raw_sentences = [
    'paris is the capital of france',
    'berlin is the capital of germany',
    'london is the capital of england',
    'rome is the capital of italy',
    'madrid is the capital of spain',
    'france and germany are countries in europe',
    'italy and spain share a common border',
    'london and paris are major european cities',
    'berlin and rome are historic european capitals',
    'madrid is a vibrant city in western europe',
    'cat is a common household pet',
    'dog is a loyal animal companion',
    'cat and dog are popular domestic animals',
    'animals live in forests rivers and cities',
    'the cat chased the mouse across the floor',
    'the dog barked at the cat near the tree',
    'one two three four five are numbers',
    'six seven eight nine ten are also numbers',
    'numbers and words are both linguistic units',
    'language models learn from many words and sentences',
]

# ── Build vocabulary ─────────────────────────────────────────────────
all_tokens = [w for sent in raw_sentences for w in sent.split()]
word_freq = Counter(all_tokens)
# Keep top-50 words for a tractable demo
VOCAB_SIZE = 50
most_common = [w for w, _ in word_freq.most_common(VOCAB_SIZE)]
w2i = {w: i for i, w in enumerate(most_common)}
i2w = {i: w for w, i in w2i.items()}
print(f'Vocabulary size: {len(w2i)}')

def make_skipgram_pairs(sentences: list, w2i: dict,
                        window: int = 2) -> list:
    """Generate (center_idx, context_idx) pairs."""
    pairs = []
    for sent in sentences:
        tokens = [w for w in sent.split() if w in w2i]
        for i, center in enumerate(tokens):
            for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
                if j != i:
                    pairs.append((w2i[center], w2i[tokens[j]]))
    return pairs

pairs = make_skipgram_pairs(raw_sentences, w2i, window=2)
print(f'(center, context) pairs: {len(pairs)}')

# ── Numpy SGD training ───────────────────────────────────────────────
EMBED_DIM = 20
LR_NP = 0.05
EPOCHS = 80

rng = np.random.default_rng(42)
W_in  = rng.normal(0, 0.1, (VOCAB_SIZE, EMBED_DIM))  # center embeddings
W_out = rng.normal(0, 0.1, (VOCAB_SIZE, EMBED_DIM))  # context embeddings

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -15, 15)))

losses = []
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    # Shuffle training pairs each epoch
    idx_order = rng.permutation(len(pairs))
    for k in idx_order:
        c_idx, ctx_idx = pairs[k]
        score = np.dot(W_in[c_idx], W_out[ctx_idx])  # positive score
        p = sigmoid(score)
        loss = -np.log(p + 1e-9)
        epoch_loss += loss
        grad = p - 1.0  # d_loss / d_score
        W_in[c_idx]   -= LR_NP * grad * W_out[ctx_idx]
        W_out[ctx_idx] -= LR_NP * grad * W_in[c_idx]
    losses.append(epoch_loss / len(pairs))

print(f'Final training loss: {losses[-1]:.4f}')

def nearest_neighbors(word: str, W: np.ndarray, w2i: dict,
                       i2w: dict, k: int = 5) -> list:
    """Find k-nearest neighbours by cosine similarity."""
    if word not in w2i:
        return []
    idx = w2i[word]
    vec = W[idx]
    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-9
    sims = W @ vec / (norms.squeeze() * (np.linalg.norm(vec) + 1e-9))
    sims[idx] = -1  # exclude self
    top_k = np.argsort(sims)[::-1][:k]
    return [(i2w[i], float(sims[i])) for i in top_k]

for probe in ['paris', 'cat', 'numbers']:
    if probe in w2i:
        nbrs = nearest_neighbors(probe, W_in, w2i, i2w, k=5)
        print(f'\nNearest to "{probe}": {[n for n, _ in nbrs]}')


## Level 2: Word2Vec with Negative Sampling in PyTorch

Negative sampling approximates the full softmax by contrasting positive (center, context)
pairs against K randomly sampled 'negative' context words. Loss:
L = -log σ(c·p) - Σ log σ(-c·n_k)

In [ ]:
class SkipGramNeg(nn.Module):
    """Skip-gram model trained with noise-contrastive estimation (neg sampling)."""
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.center_embed  = nn.Embedding(vocab_size, embed_dim)
        self.context_embed = nn.Embedding(vocab_size, embed_dim)
        # Initialise embeddings with small values for stable training
        nn.init.uniform_(self.center_embed.weight,  -0.1, 0.1)
        nn.init.uniform_(self.context_embed.weight, -0.1, 0.1)

    def forward(self, center: torch.Tensor, context: torch.Tensor,
                negatives: torch.Tensor) -> torch.Tensor:
        """Return scalar mean negative sampling loss."""
        c   = self.center_embed(center)          # [B, D]
        pos = self.context_embed(context)        # [B, D]
        neg = self.context_embed(negatives)      # [B, K, D]
        pos_score = (c * pos).sum(-1)            # [B]      dot product
        neg_score = (c.unsqueeze(1) * neg).sum(-1)  # [B, K]
        # Positive pair: maximise sigmoid score; negative: minimise
        loss = (-torch.log(torch.sigmoid(pos_score) + 1e-9)
                - torch.log(torch.sigmoid(-neg_score) + 1e-9).sum(-1))
        return loss.mean()

# ── Hyperparameters ───────────────────────────────────────────────────
EMBED_DIM_PT = 20
NEG_SAMPLES  = 5
BATCH_SIZE   = 32
STEPS        = 200

model = SkipGramNeg(VOCAB_SIZE, EMBED_DIM_PT).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Convert pairs to tensors for batched training
centers_t  = torch.tensor([p[0] for p in pairs], dtype=torch.long)
contexts_t = torch.tensor([p[1] for p in pairs], dtype=torch.long)

def sample_negatives(batch_size: int, k: int, vocab_size: int,
                     exclude: torch.Tensor) -> torch.Tensor:
    """Sample k negatives per example (uniform, may include positives—acceptable for demo)."""
    return torch.randint(0, vocab_size, (batch_size, k))

training_losses = []
n_pairs = len(pairs)

for step in range(STEPS):
    # Random mini-batch of (center, context) pairs
    batch_idx = torch.randint(0, n_pairs, (BATCH_SIZE,))
    center_b  = centers_t[batch_idx].to(device)
    context_b = contexts_t[batch_idx].to(device)
    neg_b     = sample_negatives(BATCH_SIZE, NEG_SAMPLES,
                                 VOCAB_SIZE, context_b).to(device)

    optimizer.zero_grad()
    loss = model(center_b, context_b, neg_b)
    loss.backward()
    optimizer.step()
    training_losses.append(loss.item())

print(f'Final training loss: {training_losses[-1]:.4f}')

# ── Extract trained embeddings ────────────────────────────────────────
W_pt = model.center_embed.weight.detach().cpu().numpy()

print('\nTop-5 nearest neighbours (Word2Vec negative sampling):')
for probe in ['paris', 'cat', 'france', 'numbers', 'berlin']:
    if probe in w2i:
        nbrs = nearest_neighbors(probe, W_pt, w2i, i2w, k=5)
        print(f'  {probe:10s}: {[n for n, _ in nbrs]}')

# ── Vector arithmetic ─────────────────────────────────────────────────
# paris - france + germany ≈ berlin on geographic synthetic data
for (a, b, c, expected) in [
    ('paris',  'france',  'germany', 'berlin'),
    ('london', 'england', 'spain',   'madrid'),
]:
    if all(w in w2i for w in [a, b, c]):
        v = W_pt[w2i[b]] - W_pt[w2i[a]] + W_pt[w2i[c]]
        # Find nearest word to v (exclude a, b, c)
        norms = np.linalg.norm(W_pt, axis=1) + 1e-9
        v_norm = np.linalg.norm(v) + 1e-9
        sims = W_pt @ v / (norms * v_norm)
        for excl in [w2i[a], w2i[b], w2i[c]]:
            sims[excl] = -1
        pred = i2w[int(np.argmax(sims))]
        print(f'\n{b} - {a} + {c} = {pred}  (expected: {expected})')

# ── Loss curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(training_losses, color='steelblue', alpha=0.7)
ax.set_xlabel('Training step')
ax.set_ylabel('NS loss')
ax.set_title('Word2Vec Negative Sampling Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/nlp02_loss.png', dpi=80)
plt.close()
print('\nLoss curve saved to /tmp/nlp02_loss.png')


## Real-World Example 1: Sentiment Classification — Pretrained vs Random Embeddings

We compare two strategies for initialising a text classifier's embedding layer:
1. **Random init**: embeddings start uniform random and train with the classifier.
2. **Pretrained (frozen)**: use skip-gram embeddings; only the classifier head trains.

In [ ]:
# ── Simple 1-layer embedding + mean-pooling + linear classifier ─────
class EmbeddingClassifier(nn.Module):
    """Mean-pool word embeddings then classify with linear layer."""
    def __init__(self, vocab_size: int, embed_dim: int,
                 pretrained: np.ndarray = None, freeze: bool = False):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        if pretrained is not None:
            # Load pre-trained weights from word2vec
            self.embed.weight.data.copy_(
                torch.tensor(pretrained, dtype=torch.float32))
        if freeze:
            self.embed.weight.requires_grad_(False)
        self.fc = nn.Linear(embed_dim, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, T] token indices. Returns logits [B, 2]."""
        emb = self.embed(x)         # [B, T, D]
        pooled = emb.mean(dim=1)    # [B, D]  simple mean pooling
        return self.fc(pooled)       # [B, 2]

# ── Build small sentiment dataset using in-vocab words ───────────────
pos_templates = [
    'paris is the capital of france',
    'cat is a loyal animal companion',
    'london and paris are major cities',
    'one two three four five are numbers',
    'berlin and rome are historic capitals',
]
neg_templates = [
    'the dog chased the cat near the tree',
    'animals live in forests rivers and cities',
    'language models learn from many words',
    'numbers and words are linguistic units',
    'italy and spain share a common border',
]

def encode_sentence(sent: str, w2i: dict, max_len: int = 8) -> list:
    """Convert sentence to fixed-length index list (pad with 0)."""
    tokens = [w2i.get(w, 0) for w in sent.split() if w in w2i]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))  # zero-pad
    return tokens

X_sents = pos_templates + neg_templates
y_sents = [1] * 5 + [0] * 5
X_enc = torch.tensor([encode_sentence(s, w2i) for s in X_sents], dtype=torch.long)
y_enc = torch.tensor(y_sents, dtype=torch.long)

def train_classifier(model: nn.Module, X: torch.Tensor, y: torch.Tensor,
                     epochs: int = 100, lr: float = 0.01) -> list:
    """Train classifier; return per-epoch loss list."""
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses_out = []
    for ep in range(epochs):
        opt.zero_grad()
        logits = model(X.to(device))
        loss = criterion(logits, y.to(device))
        loss.backward()
        opt.step()
        losses_out.append(loss.item())
    return losses_out

# Random init model
model_rand = EmbeddingClassifier(VOCAB_SIZE, EMBED_DIM_PT,
                                  pretrained=None, freeze=False).to(device)
losses_rand = train_classifier(model_rand, X_enc, y_enc, epochs=100)

# Pretrained (frozen embeddings) model
model_pt = EmbeddingClassifier(VOCAB_SIZE, EMBED_DIM_PT,
                                pretrained=W_pt, freeze=True).to(device)
losses_pt = train_classifier(model_pt, X_enc, y_enc, epochs=100)

def eval_acc(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    with torch.no_grad():
        logits = model(X.to(device))
        preds = logits.argmax(dim=-1).cpu()
    return accuracy_score(y.numpy(), preds.numpy())

acc_rand = eval_acc(model_rand, X_enc, y_enc)
acc_pt   = eval_acc(model_pt,   X_enc, y_enc)
print(f'Random init accuracy:    {acc_rand:.3f}')
print(f'Pretrained (frozen) acc: {acc_pt:.3f}')
print('Pretrained embeddings provide a warm start; fewer updates needed.')


## Real-World Example 2: Word Analogy Evaluation

Classic analogy evaluation: given a:b::c:?, compute v(b)-v(a)+v(c)
and check if the nearest word is the expected answer d.

In [ ]:
# ── Define 10 analogy triples using words in our vocabulary ──────────
# Format: (a, b, c, expected_d)  meaning  a:b :: c:d
# We use geographic and categorical analogies from our synthetic corpus
analogy_triples = [
    ('paris',  'france',  'germany',  'berlin'),
    ('paris',  'france',  'italy',    'rome'),
    ('paris',  'france',  'spain',    'madrid'),
    ('london', 'england', 'italy',    'rome'),
    ('london', 'england', 'germany',  'berlin'),
    ('berlin', 'germany', 'france',   'paris'),
    ('rome',   'italy',   'spain',    'madrid'),
    ('cat',    'animal',  'france',   'country'),
    ('one',    'numbers', 'france',   'country'),
    ('six',    'numbers', 'cat',      'animal'),
]

def analogy_answer(a: str, b: str, c: str, W: np.ndarray,
                   w2i: dict, i2w: dict) -> str:
    """Return the word closest to W[b] - W[a] + W[c]."""
    if not all(w in w2i for w in [a, b, c]):
        return '<OOV>'
    v = W[w2i[b]] - W[w2i[a]] + W[w2i[c]]
    norms  = np.linalg.norm(W, axis=1) + 1e-9
    v_norm = np.linalg.norm(v) + 1e-9
    sims   = W @ v / (norms * v_norm)
    # Exclude a, b, c from answers
    for excl_word in [a, b, c]:
        if excl_word in w2i:
            sims[w2i[excl_word]] = -1
    return i2w[int(np.argmax(sims))]

correct_np = 0
correct_pt = 0
total_valid = 0

print(f'{'Analogy':35s}  {'Expected':10s}  {'Numpy':10s}  {'Word2Vec':10s}')
print('-' * 75)

for (a, b, c, expected) in analogy_triples:
    if not all(w in w2i for w in [a, b, c, expected]):
        continue
    total_valid += 1
    pred_np = analogy_answer(a, b, c, W_in, w2i, i2w)
    pred_pt = analogy_answer(a, b, c, W_pt, w2i, i2w)
    correct_np += int(pred_np == expected)
    correct_pt += int(pred_pt == expected)
    label = f'{a}:{b} :: {c}:?'
    print(f'{label:35s}  {expected:10s}  {pred_np:10s}  {pred_pt:10s}')

if total_valid > 0:
    print(f'\nNumpy skip-gram accuracy: {correct_np}/{total_valid} = '
          f'{correct_np/total_valid:.1%}')
    print(f'Word2Vec negative sampling accuracy: {correct_pt}/{total_valid} = '
          f'{correct_pt/total_valid:.1%}')
print('\nNote: accuracy is low on tiny corpus — real Word2Vec trains on billions of words.')


## Real-World Example 3 + Comparison

### Example 3: t-SNE Visualization of Word Embeddings
Project 20-dimensional embeddings to 2D via sklearn TSNE.
Color by word category (geography, animals, numbers) to reveal clustering.

### Comparison: Random Init vs Word2Vec — Downstream Classification Accuracy

In [ ]:
# ── Define word categories for colouring t-SNE ───────────────────────
categories = {
    'geography': ['paris', 'berlin', 'london', 'rome', 'madrid',
                  'france', 'germany', 'england', 'italy', 'spain'],
    'animals':   ['cat', 'dog', 'animal', 'animals', 'mouse'],
    'numbers':   ['one', 'two', 'three', 'four', 'five',
                  'six', 'seven', 'eight', 'nine', 'ten'],
}

words_to_plot = []
colors_map = {'geography': 'steelblue', 'animals': 'darkorange', 'numbers': 'forestgreen'}
word_colors = []

for cat, words in categories.items():
    for w in words:
        if w in w2i:
            words_to_plot.append(w)
            word_colors.append(colors_map[cat])

if len(words_to_plot) >= 10:
    vecs_to_plot = W_pt[[w2i[w] for w in words_to_plot]]
    # t-SNE requires perplexity < n_samples
    perp = min(5, len(words_to_plot) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perp,
                n_iter=500, init='random')
    coords_2d = tsne.fit_transform(vecs_to_plot)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # t-SNE scatter
    ax = axes[0]
    for cat, col in colors_map.items():
        mask = [i for i, w in enumerate(words_to_plot)
                if w in categories[cat]]
        if mask:
            ax.scatter(coords_2d[mask, 0], coords_2d[mask, 1],
                       c=col, label=cat, s=80, alpha=0.85)
            for i in mask:
                ax.annotate(words_to_plot[i],
                            (coords_2d[i, 0], coords_2d[i, 1]),
                            fontsize=7, alpha=0.85)
    ax.set_title('t-SNE of Word2Vec Embeddings (by category)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    # ── Comparison: random vs pretrained accuracy ─────────────────────
    ax2 = axes[1]
    epoch_range = list(range(0, 100, 5))

    def losses_to_smooth(losses: list, window: int = 5) -> list:
        """Rolling mean to smooth noisy loss curves."""
        return [np.mean(losses[max(0, i-window):i+1]) for i in range(len(losses))]

    ax2.plot(losses_to_smooth(losses_rand), label='Random init',
             color='darkorange', linewidth=2)
    ax2.plot(losses_to_smooth(losses_pt),   label='Pretrained (frozen)',
             color='steelblue', linewidth=2)
    ax2.set_xlabel('Training epoch')
    ax2.set_ylabel('CrossEntropy loss')
    ax2.set_title('Classifier Loss: Random vs Pretrained Embeddings')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Word2Vec Embedding Analysis', fontsize=13)
    plt.tight_layout()
    plt.savefig('/tmp/nlp02_comparison.png', dpi=80, bbox_inches='tight')
    plt.close()
    print('t-SNE + comparison plot saved to /tmp/nlp02_comparison.png')
else:
    print('Not enough words in vocab for t-SNE — increase corpus size.')

print(f'\nFinal accuracy summary:')
print(f'  Random init:    {acc_rand:.3f}')
print(f'  Pretrained:     {acc_pt:.3f}')
print('\nKey insight: pretrained embeddings capture semantic structure')
print('even from a small corpus, helping classifiers converge faster.')
